In [33]:
from pathlib import Path
import pandas as pd

# Measured features
features = pd.read_csv(
    "../../measurements/red_channel_df.csv"
)

print(features.shape)
print(features.columns[:10])

# Read all annotation files
annotation_folder = Path("../../classification/annotations")
annotation_files = sorted(annotation_folder.glob("*.csv"))
annotations = pd.concat(
    (pd.read_csv(f) for f in annotation_files),
    ignore_index=True,
)

# Diagnostics — now both exist
print("features columns:", features.columns.tolist())
print("annotations columns:", annotations.columns.tolist())
print(f"Loaded {len(annotation_files)} annotation files.")
print("features shape:", features.shape)
print("annotations shape:", annotations.shape)

# Merge on image and label
df = features.merge(
    annotations[["label", "image", "annotation"]],
    on=["label", "image"],
    how="inner",
)
print(df.shape)
print(df["annotation"].value_counts())

(543371, 113)
Index(['image', 'label', 'intensity_mean_red', 'intensity_std_red',
       'intensity_p10_red', 'intensity_p20_red', 'intensity_p30_red',
       'intensity_p40_red', 'intensity_p50_red', 'intensity_p60_red'],
      dtype='object')
features columns: ['image', 'label', 'intensity_mean_red', 'intensity_std_red', 'intensity_p10_red', 'intensity_p20_red', 'intensity_p30_red', 'intensity_p40_red', 'intensity_p50_red', 'intensity_p60_red', 'intensity_p70_red', 'intensity_p80_red', 'intensity_p90_red', 'intensity_p92_red', 'intensity_p95_red', 'intensity_p98_red', 'red_blur_0p5um_mean', 'red_blur_0p5um_median', 'red_blur_0p5um_std', 'red_blur_0p5um_min', 'red_blur_0p5um_max', 'red_blur_0p5um_sum', 'red_blur_0p5um_cv', 'red_blur_0p5um_p10', 'red_blur_0p5um_p20', 'red_blur_0p5um_p30', 'red_blur_0p5um_p40', 'red_blur_0p5um_p50', 'red_blur_0p5um_p60', 'red_blur_0p5um_p70', 'red_blur_0p5um_p80', 'red_blur_0p5um_p90', 'red_blur_0p5um_p92', 'red_blur_0p5um_p95', 'red_blur_0p5um_p98', 'r

In [34]:
annotations

,image,point_index,annotation,label,z,y,x
0,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,0,1,1183,5.500000,477.950227,511.003367
1,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,1,1,2691,5.500000,450.323170,512.908682
2,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,2,1,3549,5.500000,511.293228,606.269087
3,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,3,1,2984,5.500000,579.884543,586.263286
4,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,4,1,2565,5.500000,672.292287,698.676835
...,...,...,...,...,...,...,...
2102,F2445_CHST11KO+P301S_MC_AT8_epsilon1_third_202...,11,0,334,8.803609,621.493022,588.301304
2103,F2445_CHST11KO+P301S_MC_AT8_epsilon1_third_202...,12,0,437,8.993426,636.471134,591.553263
2104,F2445_CHST11KO+P301S_MC_AT8_epsilon1_third_202...,13,0,659,8.339059,568.167173,599.615138
2105,F2445_CHST11KO+P301S_MC_AT8_epsilon1_third_202...,14,0,281,9.513133,637.020942,647.235256


In [35]:
import numpy as np

def is_non_binned(image_name):
    """Detect non-binned images by name pattern only."""
    name_str = str(image_name).lower()
    non_binned_patterns = ["1.3_", "nc_"]
    return any(pattern in name_str for pattern in non_binned_patterns)

# Identify intensity columns
intensity_cols = [col for col in features.columns 
                  if "intensity" in col.lower() or "mean" in col.lower()]
print(f"Intensity columns: {intensity_cols}")

# red dilated r2 shell mean and red puncta structure p99 mean size are expected to be numeric --> there was an issue with them

#features["red_dilated_r2_shell_mean"] = pd.to_numeric(
#    features["red_dilated_r2_shell_mean"],
#    errors="coerce"
#)

#features["red_puncta_structure_p99_mean_size"] = pd.to_numeric(
#    features["red_puncta_structure_p99_mean_size"],
#    errors="coerce"
#)

# Compute per-image mean intensity to validate detection
image_stats = (
    features.groupby("image")[intensity_cols]
    .mean()
    .assign(is_non_binned=lambda df: df.index.map(is_non_binned))
)
print("\nPer-image mean intensity (sample):")
print(image_stats.sort_values("is_non_binned", ascending=False).head(20))

# Confirm all images look consistent within each group
print("\nMean intensity by group:")
print(image_stats.groupby("is_non_binned")[intensity_cols].mean())


Intensity columns: ['intensity_mean_red', 'intensity_std_red', 'intensity_p10_red', 'intensity_p20_red', 'intensity_p30_red', 'intensity_p40_red', 'intensity_p50_red', 'intensity_p60_red', 'intensity_p70_red', 'intensity_p80_red', 'intensity_p90_red', 'intensity_p92_red', 'intensity_p95_red', 'intensity_p98_red', 'red_blur_0p5um_mean', 'red_blur_1p0um_mean', 'red_blur_2p0um_mean', 'red_dilated_r2_mean', 'red_dilated_r2_shell_mean', 'red_dilated_r2_inside_mean', 'red_dilated_r2_shell_to_inside_mean_ratio', 'red_dilated_r2_dilated_to_inside_mean_ratio', 'red_puncta_p99_mean_high_pixels', 'red_puncta_structure_p99_mean_size', 'red_glcm_contrast_mean', 'red_glcm_dissimilarity_mean', 'red_glcm_homogeneity_mean', 'red_glcm_ASM_mean', 'red_glcm_energy_mean', 'red_glcm_correlation_mean']

Per-image mean intensity (sample):
                                                    intensity_mean_red  \
image                                                                    
NC_31.3u-26.2.25_WT+PLA_M

In [43]:
df_train = df.dropna(subset=["annotation"])

from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_train,
    test_size=0.5,      # 20% test, 80% train
    random_state=42,    # reproducible split
    stratify=df_train["annotation"],  # preserve class balance
)

In [44]:
drop_columns = [
    "annotation",
    "label",
    "image_name",
    "valid",
]

feature_cols = [c for c in df_train.columns if c not in drop_columns]

X_train = df_train[feature_cols].select_dtypes(include="number")
X_train.fillna(X_train.median())
y_train = df_train["annotation"]

X_test = df_test[feature_cols].select_dtypes(include="number")
X_test.fillna(X_test.median())
y_test = df_test["annotation"]

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
)

model.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_,
})

importance = importance.sort_values(
    "importance",
    ascending=False,
)

print(importance.head(20))

                                         feature  importance
83                      red_puncta_p99_threshold    0.038555
94            red_puncta_structure_p99_threshold    0.034865
73                            red_dilated_r2_max    0.016081
82   red_dilated_r2_dilated_to_inside_mean_ratio    0.015463
77                      red_dilated_r2_shell_max    0.015079
17                            red_blur_0p5um_min    0.014122
100                        red_glcm_contrast_std    0.014084
76                      red_dilated_r2_shell_p95    0.013548
106                             red_glcm_ASM_std    0.013390
110                     red_glcm_correlation_std    0.013146
81     red_dilated_r2_shell_to_inside_mean_ratio    0.013042
96                 red_z_distribution_weighted_z    0.012855
38                            red_blur_1p0um_sum    0.012672
78                      red_dilated_r2_shell_sum    0.012607
109                    red_glcm_correlation_mean    0.012603
19                      

In [47]:
all_features = features.copy()

# Detect non-binned images
non_binned_mask_all = all_features["image"].apply(is_non_binned)

# Apply same normalization to all features before prediction
intensity_cols_all = [
    col for col in all_features.columns
    if "intensity" in col.lower() or "mean" in col.lower()
]

if intensity_cols_all:
    for col in intensity_cols_all:
        if non_binned_mask_all.any():
            all_features.loc[non_binned_mask_all, col] = (
                all_features.loc[non_binned_mask_all, col] / 16.0
            )

X_all = (
    all_features
    .drop(columns=[c for c in drop_columns if c in all_features.columns])
    .select_dtypes(include="number")
)

# Ensure exact same features as training
X_all = X_all.reindex(columns=X_train.columns)

# Fill missing values
X_all = X_all.fillna(X_train.median())

all_features["prediction"] = model.predict(X_all)
all_features["probability"] = model.predict_proba(X_all)[:, 1]

all_features.to_csv(
    "../../classification/output/props_with_predictions.csv",
    index=False,
)

In [ ]:
# TEST

import numpy as np
import pandas as pd
import tifffile
import napari
import skimage.transform
from bioio import BioImage

image_name = "64U-J9910_CHST11KO_BS_AT8_delta4_first_2026-04-23_12.32.20_FusionStitcher"
group = "CHST11KO"
region = "brainstem"
MASK_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/segmentation_output_pseudo3d_260611/{region}/{group}/{image_name}_input_channel.tif"
IMAGE_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/{region}/{group}/{image_name}.ims"
CSV_PATH = "../../classification/output/props_with_predictions.csv"

CH_DAPI, CH_GREEN, CH_RED, CH_FARRED = 0, 1, 2, 3
CLASSIFICATION_COLUMN = "prediction"

In [ ]:
# TEST

bio = BioImage(IMAGE_PATH)
bio.set_scene(bio.scenes[0])
image_czyx = bio.get_image_data("CZYX", T=0)
print("Image shape:", image_czyx.shape)

mask = tifffile.imread(MASK_PATH).squeeze().astype(np.uint32)
print("Mask shape:", mask.shape)

table = pd.read_csv(CSV_PATH)
print("CSV shape:", table.shape)
print("CSV columns:", table.columns.tolist())

Image shape: (4, 12, 4548, 4661)
Mask shape: (12, 4548, 4661)
CSV shape: (543371, 115)
CSV columns: ['image', 'label', 'intensity_mean_red', 'intensity_std_red', 'intensity_p10_red', 'intensity_p20_red', 'intensity_p30_red', 'intensity_p40_red', 'intensity_p50_red', 'intensity_p60_red', 'intensity_p70_red', 'intensity_p80_red', 'intensity_p90_red', 'intensity_p92_red', 'intensity_p95_red', 'intensity_p98_red', 'red_blur_0p5um_mean', 'red_blur_0p5um_median', 'red_blur_0p5um_std', 'red_blur_0p5um_min', 'red_blur_0p5um_max', 'red_blur_0p5um_sum', 'red_blur_0p5um_cv', 'red_blur_0p5um_p10', 'red_blur_0p5um_p20', 'red_blur_0p5um_p30', 'red_blur_0p5um_p40', 'red_blur_0p5um_p50', 'red_blur_0p5um_p60', 'red_blur_0p5um_p70', 'red_blur_0p5um_p80', 'red_blur_0p5um_p90', 'red_blur_0p5um_p92', 'red_blur_0p5um_p95', 'red_blur_0p5um_p98', 'red_blur_1p0um_mean', 'red_blur_1p0um_median', 'red_blur_1p0um_std', 'red_blur_1p0um_min', 'red_blur_1p0um_max', 'red_blur_1p0um_sum', 'red_blur_1p0um_cv', 'red_blu

In [ ]:
# TEST

import pandas as pd
from pathlib import Path

# Check the original per-image props CSV
original_props = pd.read_csv(f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/histograms/{image_name}_props.csv")

print("Original props columns:", original_props.columns.tolist()[:10])
print("First 10 labels:", original_props["label"].values[:10])
print("Label range:", original_props["label"].min(), "to", original_props["label"].max())
print("Shape:", original_props.shape)

Original props columns: ['image_name', 'read_mode', 'was_python_downscaled', 'label', 'area', 'num_pixels', 'dapi_positive_overlap', 'valid', 'tissue_area_um2', 'intensity_mean_dapi']
First 10 labels: [ 1  2  3  4  5  6  7  8  9 10]
Label range: 1 to 3476
Shape: (3476, 259)


In [2]:
import numpy as np
import pandas as pd
import tifffile
import napari
import skimage.transform
from bioio import BioImage
from scipy import ndimage as ndi
from magicgui import magicgui


image_name = "64U-J9910_CHST11KO_BS_AT8_delta4_first_2026-04-23_12.32.20_FusionStitcher"
group = "CHST11KO"
region = "brainstem"
MASK_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/segmentation_output_pseudo3d_260611/{region}/{group}/{image_name}_mask_3d.tif"
IMAGE_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/{region}/{group}/{image_name}.ims"

CSV_PATH = "../../classification/output/props_with_predictions.csv"

BIOIO_SCENE_INDEX = 0

CH_DAPI = 0
CH_GREEN = 1
CH_RED = 2
CH_FARRED = 3

CLASSIFICATION_COLUMN = "prediction"

# If your CSV has a continuous score/probability column (e.g. "prob_at8",
# "score", "probability", values roughly in [0, 1]), name it here to get
# an interactive threshold slider in napari. Leave as None to just color
# points by the binary CLASSIFICATION_COLUMN with no slider.
PROBABILITY_COLUMN = None  # e.g. "prob_at8"

POINT_SIZE = 12  # in mask/image pixel units -- tune to taste for visibility


def load_image_czyx(path, scene_index=0):
    bio = BioImage(str(path))
    bio.set_scene(bio.scenes[scene_index])
    return bio.get_image_data("CZYX", T=0)


def downscale_image_to_mask_shape(image_czyx, mask_shape):
    if image_czyx.shape[1:] == mask_shape:
        return image_czyx
    target_shape = (image_czyx.shape[0],) + tuple(mask_shape)
    print(f"Downscaling image from {image_czyx.shape} to {target_shape}")
    return skimage.transform.resize(
        image_czyx,
        target_shape,
        order=1,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(image_czyx.dtype, copy=False)


def make_classification_mask(mask, table, class_column):
    table = table.dropna(subset=[class_column]).copy()
    table["label"] = table["label"].astype(np.uint32)
    table[class_column] = table[class_column].astype(int)

    max_label = int(mask.max())
    lut = np.zeros(max_label + 1, dtype=np.uint8)

    for label_id, cls in zip(
        table["label"].to_numpy(),
        table[class_column].to_numpy(),
    ):
        if 0 < label_id <= max_label:
            lut[int(label_id)] = int(cls) + 1

    return lut[mask]


def compute_label_centroids(mask):
    """3D centroid (z, y, x) of every labeled cell in one pass, using
    scipy.ndimage.center_of_mass with an explicit index list. This is
    O(volume) total rather than O(n_labels x volume), which matters once
    you have hundreds/thousands of small densely-packed cells."""
    labels_present = np.unique(mask)
    labels_present = labels_present[labels_present > 0]
    if labels_present.size == 0:
        return pd.DataFrame(columns=["label", "z", "y", "x"])

    centroids = ndi.center_of_mass(mask > 0, labels=mask, index=labels_present)
    centroids = np.asarray(centroids)  # (n_labels, 3) -> z, y, x
    return pd.DataFrame({
        "label": labels_present.astype(np.uint32),
        "z": centroids[:, 0],
        "y": centroids[:, 1],
        "x": centroids[:, 2],
    })


def build_points_table(mask, table, class_column, probability_column=None):
    """Merges every labeled cell's centroid with its classification (and
    optional probability score), for use as a napari Points layer instead
    of a dense per-voxel label overlay -- much more legible with many
    small, tightly-packed cells."""
    centroids = compute_label_centroids(mask)

    keep_cols = ["label", class_column]
    if probability_column is not None and probability_column in table.columns:
        keep_cols.append(probability_column)

    merged = centroids.merge(
        table[keep_cols].dropna(subset=[class_column]),
        on="label",
        how="inner",  # only cells that made it into the classification table
    )
    merged[class_column] = merged[class_column].astype(int)
    return merged


def contrast_limits(volume, low=1, high=99.5):
    """
    Compute contrast limits from the MIP rather than the full volume —
    faster and avoids empty Z-slices pulling the percentile down.
    """
    mip = volume.max(axis=0)
    lo, hi = np.percentile(mip, [low, high])
    # Guard against flat images (all zeros, etc.)
    if hi <= lo:
        hi = lo + 1
    return [float(lo), float(hi)]


def add_channel(viewer, volume, name, colormap, visible=True):
    viewer.add_image(
        volume,
        name=name,
        colormap=colormap,
        blending="additive",
        contrast_limits=contrast_limits(volume),
        visible=visible,
    )


def add_prediction_points(viewer, points_df, class_column, probability_column=None):
    """Adds classified cells as two separate Points layers (positive /
    negative) instead of a filled label volume, so individual predictions
    stay visually distinguishable even with many densely-packed cells.
    Points are shown as small rings at each cell's 3D centroid and are
    visible across a few nearby Z-slices (out_of_slice_display) so you
    don't lose them between slider ticks.
    """
    positive = points_df[points_df[class_column] == 1]
    negative = points_df[points_df[class_column] == 0]

    pos_coords = positive[["z", "y", "x"]].to_numpy()
    neg_coords = negative[["z", "y", "x"]].to_numpy()

    viewer.add_points(
        pos_coords,
        name="AT8+ NeuN (predicted)",
        symbol="ring",
        size=POINT_SIZE,
        face_color="transparent",
        border_color="red",
        border_width=0.15,
        out_of_slice_display=True,
        visible=True,
    )

    viewer.add_points(
        neg_coords,
        name="AT8- NeuN (predicted)",
        symbol="ring",
        size=POINT_SIZE,
        face_color="transparent",
        border_color="cyan",
        border_width=0.15,
        out_of_slice_display=True,
        visible=True,
    )

    print(f"  {len(positive)} AT8+ points, {len(negative)} AT8- points added.")

    if probability_column is not None and probability_column in points_df.columns:
        add_probability_threshold_widget(viewer, points_df, probability_column)


def add_probability_threshold_widget(viewer, points_df, probability_column):
    """Interactive slider: only shows cells with probability >= threshold,
    colored red/cyan by their binary prediction as before. Lets you see
    exactly which borderline cells drop in/out as you move the threshold."""
    prob_min = float(points_df[probability_column].min())
    prob_max = float(points_df[probability_column].max())

    all_coords = points_df[["z", "y", "x"]].to_numpy()
    all_preds = points_df[CLASSIFICATION_COLUMN].to_numpy()
    all_probs = points_df[probability_column].to_numpy()

    threshold_layer = viewer.add_points(
        all_coords,
        name=f"thresholded by {probability_column}",
        symbol="disc",
        size=POINT_SIZE,
        face_color=np.where(all_preds == 1, "red", "cyan"),
        out_of_slice_display=True,
        visible=True,
    )

    @magicgui(
        auto_call=True,
        threshold={
            "widget_type": "FloatSlider",
            "min": prob_min,
            "max": prob_max,
            "step": (prob_max - prob_min) / 200 if prob_max > prob_min else 0.01,
        },
    )
    def threshold_widget(threshold: float = prob_min):
        keep = all_probs >= threshold
        threshold_layer.data = all_coords[keep]
        threshold_layer.face_color = np.where(all_preds[keep] == 1, "red", "cyan")
        threshold_layer.name = f"thresholded by {probability_column} (>= {threshold:.3f}, n={keep.sum()})"

    viewer.window.add_dock_widget(threshold_widget, area="right", name="Probability threshold")


def main():
    print("Loading image...")
    image_czyx = load_image_czyx(IMAGE_PATH, BIOIO_SCENE_INDEX)

    print("Loading mask...")
    mask = tifffile.imread(MASK_PATH).squeeze().astype(np.uint32)

    image_czyx = downscale_image_to_mask_shape(image_czyx, mask.shape)

    print("Loading classification table...")
    table = pd.read_csv(CSV_PATH)

    if CLASSIFICATION_COLUMN not in table.columns:
        raise ValueError(
            f"Column '{CLASSIFICATION_COLUMN}' not found. "
            f"Available: {list(table.columns)}"
        )

    if PROBABILITY_COLUMN is not None and PROBABILITY_COLUMN not in table.columns:
        print(f"  WARNING: PROBABILITY_COLUMN='{PROBABILITY_COLUMN}' not found in CSV "
              f"(available: {list(table.columns)}); skipping threshold slider.")

    # Filter table to this image only
    if "image_name" in table.columns:
        table = table[table["image_name"] == image_name]
        print(f"  {len(table)} cells for this image after filtering.")

    print("Computing cell centroids for point display...")
    points_df = build_points_table(mask, table, CLASSIFICATION_COLUMN, PROBABILITY_COLUMN)
    print(f"  {len(points_df)} classified cells with centroids computed.")

    viewer = napari.Viewer(title=image_name)

    # --- Raw channels: start with only green + red visible ---
    add_channel(viewer, image_czyx[CH_DAPI],   "DAPI",    "blue",    visible=False)
    add_channel(viewer, image_czyx[CH_GREEN],  "green",   "green",   visible=True)
    add_channel(viewer, image_czyx[CH_RED],    "red",     "red",     visible=True)
    add_channel(viewer, image_czyx[CH_FARRED], "far-red", "magenta", visible=False)

    # --- MIP layers (2D, always useful for overview) ---
    viewer.add_image(
        image_czyx[CH_RED].max(axis=0),
        name="MIP red",
        colormap="red",
        blending="additive",
        contrast_limits=contrast_limits(image_czyx[CH_RED]),
        visible=False,
    )

    viewer.add_image(
        image_czyx[CH_GREEN].max(axis=0),
        name="MIP green",
        colormap="green",
        blending="additive",
        contrast_limits=contrast_limits(image_czyx[CH_GREEN]),
        visible=False,
    )

    # --- Prediction points (replaces the dense per-voxel label overlay,
    # which becomes an indistinguishable blur once you have many small,
    # tightly-packed cells) ---
    add_prediction_points(viewer, points_df, CLASSIFICATION_COLUMN, PROBABILITY_COLUMN)

    # --- Dense classification overlays kept, but OFF by default now that
    # points are the primary way to inspect predictions. Still useful for
    # checking a specific cell's exact boundary once you've zoomed in. ---
    classification_mask = make_classification_mask(mask, table, CLASSIFICATION_COLUMN)

    viewer.add_labels(
        classification_mask,
        name=f"classification 3D: {CLASSIFICATION_COLUMN}",
        opacity=0.55,
        visible=False,
    )

    viewer.add_labels(
        classification_mask.max(axis=0).astype(np.uint32),
        name=f"classification MIP: {CLASSIFICATION_COLUMN}",
        opacity=0.55,
        visible=False,
    )

    # --- Original mask (hidden by default) ---
    viewer.add_labels(
        mask,
        name="original cell labels",
        opacity=0.2,
        visible=False,
    )

    # Start at the middle Z-slice
    mid_z = mask.shape[0] // 2
    viewer.dims.set_point(0, mid_z)

    print(f"Opened napari at Z={mid_z}/{mask.shape[0]-1}. "
          f"Use the slider to navigate Z.")

    napari.run()


if __name__ == "__main__":
    main()

Loading image...
Loading mask...
Loading classification table...
Computing cell centroids for point display...
  426200 classified cells with centroids computed.
  17809 AT8+ points, 408391 AT8- points added.
Opened napari at Z=6/11. Use the slider to navigate Z.


In [ ]:
import numpy as np
import pandas as pd
import tifffile
import napari
import skimage.transform
from bioio import BioImage
from scipy import ndimage as ndi
from magicgui import magicgui

image_name = "64U-J9910_CHST11KO_BS_AT8_delta4_first_2026-04-23_12.32.20_FusionStitcher"
group = "CHST11KO"
region = "brainstem"
MASK_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/segmentation_output_pseudo3d_260611/{region}/{group}/{image_name}_mask_3d.tif"
IMAGE_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/{region}/{group}/{image_name}.ims"


# image_name = "0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04-24_13.59.05_FusionStitcher"
# group = "CHST11KO+P301S"
# region = "brainstem"
# MASK_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/segmentation_output_pseudo3d_260611/{region}/{group}/{image_name}_mask_3d.tif"
# IMAGE_PATH = f"N:/Sarah/CHST11KO&P301S project/DAPI, 488 NeuN, 594 AT8, 647 WFA/{region}/{group}/{image_name}.ims"

CSV_PATH = "../../classification/output/props_with_predictions.csv"

BIOIO_SCENE_INDEX = 0

CH_DAPI = 0
CH_GREEN = 1
CH_RED = 2
CH_FARRED = 3

CLASSIFICATION_COLUMN = "prediction"


def load_image_czyx(path, scene_index=0):
    bio = BioImage(str(path))
    bio.set_scene(bio.scenes[scene_index])
    return bio.get_image_data("CZYX", T=0)


def downscale_image_to_mask_shape(image_czyx, mask_shape):
    if image_czyx.shape[1:] == mask_shape:
        return image_czyx

    target_shape = (image_czyx.shape[0],) + tuple(mask_shape)

    print(f"Downscaling image from {image_czyx.shape} to {target_shape}")

    return skimage.transform.resize(
        image_czyx,
        target_shape,
        order=1,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(image_czyx.dtype, copy=False)


def make_classification_mask(mask, table, class_column):
    table = table.dropna(subset=[class_column]).copy()

    table = table[table["image"] == image_name]

    table["label"] = table["label"].astype(np.uint32)
    table[class_column] = table[class_column].astype(int)
    max_label = int(mask.max())
    lut = np.zeros(max_label + 1, dtype=np.uint8)

    for label_id, cls in zip(
        table["label"].to_numpy(),
        table[class_column].to_numpy(),
    ):
        if 0 < label_id <= max_label:
            # background = 0
            # class 0 = 1
            # class 1 = 2
            lut[int(label_id)] = int(cls) + 1

    return lut[mask]


def add_channel(viewer, image, name, colormap):
    viewer.add_image(
        image,
        name=name,
        colormap=colormap,
        blending="additive",
        contrast_limits=np.percentile(image, [1, 99.5]),
    )


def main():
    image_czyx = load_image_czyx(IMAGE_PATH, BIOIO_SCENE_INDEX)

    mask = tifffile.imread(MASK_PATH).squeeze().astype(np.uint32)

    image_czyx = downscale_image_to_mask_shape(
        image_czyx,
        mask.shape,
    )

    table = pd.read_csv(CSV_PATH)

    if CLASSIFICATION_COLUMN not in table.columns:
        raise ValueError(
            f"Column '{CLASSIFICATION_COLUMN}' not found in table. "
            f"Available columns are: {list(table.columns)}"
        )

    classification_mask = make_classification_mask(
        mask,
        table,
        CLASSIFICATION_COLUMN,
    )

    viewer = napari.Viewer()

    add_channel(viewer, image_czyx[CH_DAPI], "DAPI", "blue")
    add_channel(viewer, image_czyx[CH_GREEN], "green", "green")
    add_channel(viewer, image_czyx[CH_RED], "red", "red")
    add_channel(viewer, image_czyx[CH_FARRED], "far-red", "magenta")

    viewer.add_labels(
        classification_mask,
        name=f"classification: {CLASSIFICATION_COLUMN}",
        opacity=0.55,
    )

    viewer.add_labels(
        np.max(classification_mask, axis=0).astype(np.uint32),
        name=f"classification: {CLASSIFICATION_COLUMN} (max projection)",
        opacity=0.55,
    )

    viewer.add_image(
        np.max(image_czyx[CH_RED], axis=0),
        name=f"MIP red",
        colormap="red",
        blending="additive",
        contrast_limits=np.percentile(image_czyx[CH_RED], [1, 99.5]),
    )

    viewer.add_labels(
        mask,
        name="original cell labels",
        opacity=0.2,
        visible=False,
    )

    napari.run()


if __name__ == "__main__":
    main()

UniqueCountsResult(values=array([0, 1, 2], dtype=uint8), counts=array([240625491,  13700840,     52405]))


In [6]:
df = pd.read_csv(CSV_PATH)

In [7]:
df.head()

,image,label,intensity_mean_red,intensity_std_red,intensity_p10_red,intensity_p20_red,intensity_p30_red,intensity_p40_red,intensity_p50_red,intensity_p60_red,...,red_glcm_homogeneity_mean,red_glcm_homogeneity_std,red_glcm_ASM_mean,red_glcm_ASM_std,red_glcm_energy_mean,red_glcm_energy_std,red_glcm_correlation_mean,red_glcm_correlation_std,prediction,probability
0,0246_CHST11KO_BS_AT8_epsilon5_first_2026-04-24...,1,214.269323,74.233065,121.0,134.0,165.0,195.0,217.0,233.0,...,0.458725,0.083374,0.038147,0.011476,0.192876,0.030762,0.793874,0.133526,0,0.311333
1,0246_CHST11KO_BS_AT8_epsilon5_first_2026-04-24...,2,221.887827,79.548309,117.0,125.0,158.0,199.0,232.0,256.0,...,0.416450,0.066163,0.051835,0.015065,0.224903,0.035407,0.700291,0.146155,0,0.474000
2,0246_CHST11KO_BS_AT8_epsilon5_first_2026-04-24...,3,137.608778,24.982785,114.0,118.0,121.0,124.0,129.0,137.0,...,0.459821,0.087630,0.058343,0.014275,0.239510,0.031274,0.863347,0.095839,0,0.326167
3,0246_CHST11KO_BS_AT8_epsilon5_first_2026-04-24...,4,181.327679,61.296089,115.0,119.0,124.0,138.0,166.5,197.0,...,0.313775,0.064911,0.030379,0.009888,0.171631,0.030355,0.712592,0.152849,0,0.352000
4,0246_CHST11KO_BS_AT8_epsilon5_first_2026-04-24...,5,117.090595,6.471247,109.0,112.0,113.0,115.0,117.0,118.0,...,0.269116,0.038458,0.025875,0.009710,0.157476,0.032808,0.478386,0.111113,0,0.124800


In [8]:
df_filtered = df[df['image'] == image_name]

In [9]:
df_filtered.head()

,image,label,intensity_mean_red,intensity_std_red,intensity_p10_red,intensity_p20_red,intensity_p30_red,intensity_p40_red,intensity_p50_red,intensity_p60_red,...,red_glcm_homogeneity_mean,red_glcm_homogeneity_std,red_glcm_ASM_mean,red_glcm_ASM_std,red_glcm_energy_mean,red_glcm_energy_std,red_glcm_correlation_mean,red_glcm_correlation_std,prediction,probability
38472,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,1,142.346415,39.897532,112.0,114.0,116.0,119.0,122.0,132.0,...,0.488696,0.065070,0.082859,0.011959,0.287052,0.021453,0.881733,0.076298,0,0.369500
38473,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,2,218.457107,61.828020,141.0,160.0,179.0,198.0,216.0,232.4,...,0.475711,0.071708,0.089433,0.021683,0.296559,0.038542,0.802491,0.112661,0,0.281500
38474,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,3,153.041667,19.240176,128.0,135.0,141.0,147.0,153.0,158.0,...,0.558922,0.058496,0.234308,0.038047,0.482272,0.041500,0.769756,0.135239,0,0.193333
38475,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,4,113.883593,3.672952,109.0,111.0,112.0,113.0,114.0,115.0,...,0.337579,0.048809,0.053327,0.015986,0.227959,0.036902,0.512758,0.149261,0,0.314250
38476,0248_CHST11KO+P301S_BS_AT8_zeta2_first_2026-04...,5,132.813439,18.102867,117.0,120.0,123.0,125.0,128.0,131.0,...,0.376196,0.057792,0.026046,0.008355,0.159026,0.027512,0.786094,0.100407,0,0.276833


In [13]:
df_filtered.groupby('prediction').size()

prediction
0    1371
1     890
dtype: int64

In [15]:
image_name = "64U-J9910_CHST11KO_BS_AT8_delta4_first_2026-04-23_12.32.20_FusionStitcher"
df_filtered = df[df['image'] == image_name]
df_filtered.groupby('prediction').size()

prediction
0    3466
1      10
dtype: int64